In [90]:
import os
from torch import optim, nn, utils, Tensor, tensor, float32
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
import lightning as L
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torchmetrics.classification import BinaryAccuracy
from lightning.pytorch.loggers import MLFlowLogger
import numpy as np 
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [91]:
import mlflow

In [92]:
encoder = nn.Sequential(nn.Linear(4, 64), nn.Dropout(0.2), nn.ReLU(), nn.Linear(64, 8))
classifier = nn.Sequential(nn.Linear(8, 64), nn.Dropout(0.2), nn.ReLU(), nn.Linear(64, 1))

In [93]:
class LitTitanicClassifier(L.LightningModule):
    def __init__(self, encoder, classifier):
        super().__init__()
        self.encoder = encoder
        self.classifier = classifier

        self.accuracy = BinaryAccuracy()

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        # it is independent of forward
        x, y = batch
        z = self.encoder(x)
        y_hat = self.classifier(z).squeeze()

        loss = F.binary_cross_entropy_with_logits(y_hat, y)
        acc = self.accuracy(y_hat, y)
        
        self.log("train_loss", loss)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        # this is the validation loop
        x, y = batch
        z = self.encoder(x)
        y_hat = self.classifier(z).squeeze()

        val_loss = F.binary_cross_entropy_with_logits(y_hat, y)
        val_acc = self.accuracy(y_hat, y)

        self.log("val_loss", val_loss)
        self.log("val_acc", val_acc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        # this is the test loop
        x, y = batch
        z = self.encoder(x)
        y_hat = self.classifier(z).squeeze()
        
        test_loss = F.binary_cross_entropy_with_logits(y_hat, y)
        test_acc = self.accuracy(y_hat, y)
        
        self.log("test_loss", test_loss)
        self.log("test_acc", test_acc)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

In [94]:
autoencoder = LitTitanicClassifier(encoder, classifier)

In [95]:
mlf_logger = MLFlowLogger(
    experiment_name="Titanic_Survival_Prediction", # Le nom de votre projet
    tracking_uri="file:./mlruns"                   # Le dossier où tout sera sauvegardé
)

In [96]:
dataset = pd.read_csv('../../datasets/titanic/train.csv')
dataset_X_test = pd.read_csv('../../datasets/titanic/test.csv')
dataset_Y_test = pd.read_csv('../../datasets/titanic/gender_submission.csv')

In [97]:
dataset = dataset.drop(['PassengerId', 'Name', 'SibSp', 'Parch', 'Ticket', 'Cabin', 'Embarked'], axis=1)
dataset_X_test = dataset_X_test.drop(['PassengerId', 'Name', 'SibSp', 'Parch', 'Ticket', 'Cabin', 'Embarked'], axis=1)
print(dataset.iloc[0:10,:])

   Survived  Pclass     Sex   Age     Fare
0         0       3    male  22.0   7.2500
1         1       1  female  38.0  71.2833
2         1       3  female  26.0   7.9250
3         1       1  female  35.0  53.1000
4         0       3    male  35.0   8.0500
5         0       3    male   NaN   8.4583
6         0       1    male  54.0  51.8625
7         0       3    male   2.0  21.0750
8         1       3  female  27.0  11.1333
9         1       2  female  14.0  30.0708


In [98]:
dataset['Age'] = dataset['Age'].fillna(dataset['Age'].median())
dataset_X_test['Age'] = dataset_X_test['Age'].fillna(dataset_X_test['Age'].median())
dataset['Fare'] = dataset['Fare'].fillna(dataset['Fare'].median())
dataset_X_test['Fare'] = dataset_X_test['Fare'].fillna(dataset_X_test['Fare'].median())

In [99]:
dataset['Sex'] = np.where(dataset['Sex'] == 'male', 0, 1)
dataset_X_test['Sex'] = np.where(dataset_X_test['Sex'] == 'male', 0, 1)

In [100]:
X = dataset.iloc[:, 1:5].values
Y = dataset.iloc[:, 0].values
X_train, X_val, Y_train, Y_val = train_test_split(
    X, 
    Y, 
    test_size=0.2,     # 20% des données vont dans le jeu de validation
    random_state=42    # Règle d'or : toujours fixer le hasard pour avoir les mêmes résultats à chaque exécution
)
X_test = dataset_X_test.values
Y_test = dataset_Y_test['Survived'].values

In [101]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [102]:
X_tensor_train = tensor(X_train_scaled, dtype=float32)
y_tensor_train = tensor(Y_train, dtype=float32)
tensor_dataset_train = TensorDataset(X_tensor_train, y_tensor_train)
train_loader = DataLoader(tensor_dataset_train, batch_size=32, shuffle=True)

In [103]:
X_tensor_val = tensor(X_val_scaled, dtype=float32)
Y_tensor_val = tensor(Y_val, dtype=float32)
tensor_dataset_val = TensorDataset(X_tensor_val, Y_tensor_val)
val_loader = DataLoader(tensor_dataset_val, batch_size=32, shuffle=False)

In [104]:
from gc import callbacks


from lightning.pytorch.callbacks import EarlyStopping
callbacks = [EarlyStopping(monitor="train_acc", patience=5, mode="max")]
trainer = L.Trainer(callbacks=callbacks, limit_train_batches=100, max_epochs=100, logger=mlf_logger)
trainer.fit(model=autoencoder, train_dataloaders=train_loader, val_dataloaders=val_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder    │ Sequential     │    840 │ train │     0 │
│ 1 │ classifier │ Sequential     │    641 │ train │     0 │
│ 2 │ accuracy   │ BinaryAccuracy │      0 │ train │     0 │
└───┴────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 1.5 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.5 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 11                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/dagudelo/.conda/envs/tf/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:4
34: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the
`num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.

/home/dagudelo/.conda/envs/tf/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:4
34: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.

/home/dagudelo/.conda/envs/tf/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of 
training batches (23) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for 
log_every_n_steps if you want to see logs for the training epoch.

In [105]:
X_tensor_test = tensor(X_test_scaled, dtype=float32)
Y_tensor_test = tensor(Y_test, dtype=float32)
tensor_dataset_test = TensorDataset(X_tensor_test, Y_tensor_test)
test_loader = DataLoader(tensor_dataset_test, batch_size=32, shuffle=False)

In [106]:
# test the model
trainer.test(model=autoencoder, dataloaders=test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9712918400764465     │
│         test_loss         │    0.26057326793670654    │
└───────────────────────────┴───────────────────────────┘

/home/dagudelo/.conda/envs/tf/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


[{'test_loss': 0.26057326793670654, 'test_acc': 0.9712918400764465}]